# Credit Risk Assessment — Multi-Agent Parallel Analysis with MCMC Confidence

## Overview

This notebook builds a **production-grade credit risk pipeline** using the Multigen / `agentic_codex` framework. Every external service call (credit bureau, income verifier, fraud screener) is replaced with a deterministic **mock** so the entire notebook runs offline without any API keys.

### What you will learn

| Concept | Where used |
|---------|------------|
| `AgentBuilder` + `FunctionAdapter` | Defining mock agents with no real LLM | 
| `MapReduceCoordinator` | Running three analysis agents concurrently |
| `AssemblyCoordinator` | Sequencing ingest → analysis → aggregate → decision |
| `GuildRouter` | Routing approved / escalated / manual review |
| `GuardrailSandwich` | Circuit-breaker wrapper around external service calls |
| MCMC-style sampling loop | Iterating on borderline cases until consensus |
| Ensemble (`chains=5`) | Getting stable confidence scores from repeated runs |
| Audit trail | Full decision log attached to every application |

### Architecture

```
ApplicationIngestor
       │
       ▼
  ┌────────────────────────────────┐
  │       Parallel Analysis        │
  │  CreditBureauAgent  (mock)     │
  │  IncomeVerifier     (mock)     │
  │  FraudScreener      (mock)     │
  └──────────────┬─────────────────┘
                 │
                 ▼
          RiskAggregator
                 │
          ┌──────┴──────┐
          ▼             ▼             ▼
       approve      escalate    manual_review
```

---

> **Note on real LLM usage**: every agent here uses `FunctionAdapter` — a pure-Python callable that returns deterministic outputs. In production you would swap in `EnvOpenAIAdapter(model='gpt-4o')` or any LLM adapter of your choice, and add domain-specific prompts. The structural code (coordinators, routing, state machine, ensemble) remains identical.

## Section 1 — Environment Setup & Imports

We import only standard-library and `agentic_codex` dependencies. No OpenAI key needed.

In [ ]:
import json
import random
import time
import copy
import statistics
from datetime import datetime, timezone
from typing import Any, Dict, List, Optional
from dataclasses import dataclass, field

# agentic_codex core primitives
from agentic_codex import (
    AgentBuilder,
    Context,
    FunctionAdapter,
)
from agentic_codex.core.schemas import AgentStep, Message
from agentic_codex.patterns import (
    AssemblyCoordinator, Stage,
    MapReduceCoordinator,
    GuildRouter,
    GuardrailSandwich,
)

random.seed(42)   # reproducible mock outputs
print('Imports OK — no API key required.')

## Section 2 — Mock Application Data

Each applicant is a Python dict with realistic fields. We define five applicants spanning the approval spectrum:

| Applicant | Income | Credit Score | Fraud Flags | Expected Decision |
|-----------|--------|-------------|-------------|-------------------|
| Alice Chen | $120,000 | 780 | 0 | APPROVE |
| Bob Martinez | $48,000 | 610 | 1 | MANUAL_REVIEW |
| Carol Wang | $85,000 | 720 | 0 | APPROVE |
| Dave Patel | $32,000 | 540 | 2 | DECLINE |
| Eve Johnson | $65,000 | 665 | 0 | BORDERLINE (MCMC) |

In [ ]:
APPLICANTS = [
    {
        "application_id": "APP-001",
        "name": "Alice Chen",
        "annual_income": 120_000,
        "credit_score": 780,
        "debt_to_income": 0.18,
        "employment_years": 8,
        "loan_amount": 250_000,
        "loan_purpose": "mortgage",
        "fraud_flags": 0,
        "prior_defaults": 0,
    },
    {
        "application_id": "APP-002",
        "name": "Bob Martinez",
        "annual_income": 48_000,
        "credit_score": 610,
        "debt_to_income": 0.42,
        "employment_years": 2,
        "loan_amount": 80_000,
        "loan_purpose": "auto",
        "fraud_flags": 1,
        "prior_defaults": 1,
    },
    {
        "application_id": "APP-003",
        "name": "Carol Wang",
        "annual_income": 85_000,
        "credit_score": 720,
        "debt_to_income": 0.25,
        "employment_years": 5,
        "loan_amount": 150_000,
        "loan_purpose": "home_improvement",
        "fraud_flags": 0,
        "prior_defaults": 0,
    },
    {
        "application_id": "APP-004",
        "name": "Dave Patel",
        "annual_income": 32_000,
        "credit_score": 540,
        "debt_to_income": 0.61,
        "employment_years": 1,
        "loan_amount": 50_000,
        "loan_purpose": "personal",
        "fraud_flags": 2,
        "prior_defaults": 2,
    },
    {
        "application_id": "APP-005",
        "name": "Eve Johnson",
        "annual_income": 65_000,
        "credit_score": 665,
        "debt_to_income": 0.33,
        "employment_years": 4,
        "loan_amount": 120_000,
        "loan_purpose": "mortgage",
        "fraud_flags": 0,
        "prior_defaults": 0,
    },
]

print(f'Loaded {len(APPLICANTS)} applicants.')
for a in APPLICANTS:
    print(f"  {a['application_id']}: {a['name']:15s}  score={a['credit_score']}  income=${a['annual_income']:,}  fraud_flags={a['fraud_flags']}")

## Section 3 — Individual Agent Definitions

Each agent is built with `AgentBuilder` + a `FunctionAdapter` step. The `FunctionAdapter` takes a plain Python function — no LLM call.

### 3.1 ApplicationIngestor

Validates and normalises the raw application payload. In production this would parse PDFs, validate SSNs, and enrich from internal databases.

In [ ]:
def _ingest_step(ctx: Context) -> AgentStep:
    """Validate and normalise application data.
    
    REAL LLM VERSION: would send raw document text to GPT-4o with a
    structured-output schema to extract fields and detect anomalies.
    """
    app = ctx.scratch.get('application', {})
    
    # Compute derived metrics
    loan_to_income = app.get('loan_amount', 0) / max(app.get('annual_income', 1), 1)
    
    normalised = {
        **app,
        'loan_to_income_ratio': round(loan_to_income, 3),
        'ingest_timestamp': datetime.now(timezone.utc).isoformat(),
        'validation_passed': True,
    }
    
    return AgentStep(
        out_messages=[Message(role='assistant', content=json.dumps(normalised))],
        state_updates={'normalised_application': normalised},
        stop=False,
    )


ingestor_agent = (
    AgentBuilder(name='ApplicationIngestor', role='data-processor')
    .with_step(_ingest_step)
    .build()
)

print('ApplicationIngestor agent built.')

### 3.2 CreditBureauAgent (mock)

In production this calls Equifax/Experian/TransUnion APIs. Here we derive a bureau score from the applicant's `credit_score` field with synthetic noise.

In [ ]:
def _bureau_step(ctx: Context) -> AgentStep:
    """Mock credit bureau lookup.
    
    REAL VERSION: calls Equifax Connect API, parses tradelines,
    computes derogatory marks, utilisation ratio, account age.
    """
    app = ctx.scratch.get('normalised_application', ctx.scratch.get('application', {}))
    credit_score = app.get('credit_score', 600)
    prior_defaults = app.get('prior_defaults', 0)
    
    # Compute a bureau risk score (0-100, lower=riskier)
    base_score = (credit_score - 300) / 550  # normalise FICO to 0-1
    default_penalty = prior_defaults * 0.12
    bureau_risk = max(0.0, min(1.0, base_score - default_penalty))
    
    result = {
        'agent': 'CreditBureauAgent',
        'bureau_risk_score': round(bureau_risk, 4),
        'tradelines_checked': random.randint(8, 25),
        'derogatory_marks': prior_defaults,
        'credit_score_verified': credit_score,
        'report_id': f"CBR-{random.randint(100000, 999999)}",
    }
    
    return AgentStep(
        out_messages=[Message(role='assistant', content=json.dumps(result))],
        state_updates={'bureau_result': result},
        stop=False,
    )


bureau_agent = (
    AgentBuilder(name='CreditBureauAgent', role='credit-analyst')
    .with_step(_bureau_step)
    .build()
)
print('CreditBureauAgent built.')

### 3.3 IncomeVerifier (mock)

In production this calls Plaid/Finicity for bank statement analysis, or IRS Income Verification Express Service.

In [ ]:
def _income_step(ctx: Context) -> AgentStep:
    """Mock income verification.
    
    REAL VERSION: pulls 12 months of bank statements via Plaid,
    averages monthly deposits, flags large one-off transfers.
    """
    app = ctx.scratch.get('normalised_application', ctx.scratch.get('application', {}))
    stated_income = app.get('annual_income', 0)
    dti = app.get('debt_to_income', 0.3)
    employment_years = app.get('employment_years', 0)
    
    # Income verification confidence based on employment stability
    stability = min(1.0, employment_years / 5.0)
    verified_income = stated_income * (0.90 + stability * 0.10)  # small noise
    income_risk = 1.0 - (1.0 - dti) * stability  # higher DTI + less stable = higher risk
    
    result = {
        'agent': 'IncomeVerifier',
        'stated_income': stated_income,
        'verified_income': round(verified_income, 2),
        'income_match_pct': round(verified_income / stated_income * 100, 1) if stated_income > 0 else 0,
        'debt_to_income': dti,
        'income_risk_score': round(income_risk, 4),
        'employment_stability': round(stability, 4),
        'verification_method': 'bank_statement_analysis',
    }
    
    return AgentStep(
        out_messages=[Message(role='assistant', content=json.dumps(result))],
        state_updates={'income_result': result},
        stop=False,
    )


income_agent = (
    AgentBuilder(name='IncomeVerifier', role='income-analyst')
    .with_step(_income_step)
    .build()
)
print('IncomeVerifier built.')

### 3.4 FraudScreener (mock)

In production calls FICO Falcon, LexisNexis, or an internal ML model trained on historical fraud patterns.

In [ ]:
def _fraud_step(ctx: Context) -> AgentStep:
    """Mock fraud screening.
    
    REAL VERSION: calls FICO Falcon API, cross-references with
    known fraud rings, checks velocity patterns, device fingerprinting.
    """
    app = ctx.scratch.get('normalised_application', ctx.scratch.get('application', {}))
    fraud_flags = app.get('fraud_flags', 0)
    
    # Compute fraud probability
    base_fraud_prob = fraud_flags * 0.30
    noise = random.gauss(0, 0.03)
    fraud_probability = max(0.0, min(1.0, base_fraud_prob + noise))
    
    fraud_risk = {
        'agent': 'FraudScreener',
        'fraud_probability': round(fraud_probability, 4),
        'fraud_flags_detected': fraud_flags,
        'identity_verified': fraud_flags == 0,
        'velocity_check_passed': fraud_flags < 2,
        'synthetic_identity_score': round(fraud_probability * 0.8, 4),
        'screening_model': 'mock_falcon_v2',
        'risk_tier': 'LOW' if fraud_probability < 0.2 else ('MEDIUM' if fraud_probability < 0.5 else 'HIGH'),
    }
    
    return AgentStep(
        out_messages=[Message(role='assistant', content=json.dumps(fraud_risk))],
        state_updates={'fraud_result': fraud_risk},
        stop=False,
    )


fraud_agent = (
    AgentBuilder(name='FraudScreener', role='fraud-analyst')
    .with_step(_fraud_step)
    .build()
)
print('FraudScreener built.')

### 3.5 RiskAggregator

Combines the three sub-scores into a composite risk score and emits a preliminary decision with confidence rating.

In [ ]:
# Risk weight configuration (sum = 1.0)
RISK_WEIGHTS = {
    'bureau': 0.40,
    'income': 0.35,
    'fraud': 0.25,
}

def _aggregate_step(ctx: Context) -> AgentStep:
    """Weighted combination of risk signals from all three sub-agents."""
    bureau = ctx.scratch.get('bureau_result', {})
    income = ctx.scratch.get('income_result', {})
    fraud  = ctx.scratch.get('fraud_result', {})
    app    = ctx.scratch.get('normalised_application', ctx.scratch.get('application', {}))
    
    bureau_risk = bureau.get('bureau_risk_score', 0.5)
    income_risk = income.get('income_risk_score', 0.5)
    fraud_risk  = fraud.get('fraud_probability', 0.5)
    
    # Lower score = lower risk
    composite_risk = (
        (1.0 - bureau_risk) * RISK_WEIGHTS['bureau'] +   # invert: high bureau score = low risk
        income_risk          * RISK_WEIGHTS['income'] +
        fraud_risk           * RISK_WEIGHTS['fraud']
    )
    
    # Decision thresholds
    if composite_risk < 0.25:
        decision = 'APPROVE'
        confidence = 1.0 - composite_risk
    elif composite_risk < 0.45:
        decision = 'MANUAL_REVIEW'
        confidence = 0.5 + (0.45 - composite_risk) * 2
    else:
        decision = 'DECLINE'
        confidence = composite_risk
    
    result = {
        'application_id': app.get('application_id', 'UNKNOWN'),
        'applicant_name': app.get('name', ''),
        'composite_risk_score': round(composite_risk, 4),
        'bureau_contribution': round(bureau_risk, 4),
        'income_contribution': round(income_risk, 4),
        'fraud_contribution': round(fraud_risk, 4),
        'preliminary_decision': decision,
        'decision_confidence': round(confidence, 4),
        'rationale': (
            f"Bureau risk: {bureau_risk:.2f}, Income risk: {income_risk:.2f}, "
            f"Fraud probability: {fraud_risk:.2f}. "
            f"Composite score {composite_risk:.3f} → {decision}"
        ),
    }
    
    return AgentStep(
        out_messages=[Message(role='assistant', content=json.dumps(result))],
        state_updates={'aggregated_result': result},
        stop=True,
    )


aggregator_agent = (
    AgentBuilder(name='RiskAggregator', role='risk-aggregator')
    .with_step(_aggregate_step)
    .build()
)
print('RiskAggregator built.')

## Section 4 — Pipeline Assembly

We assemble the pipeline using two coordinator patterns:

1. **`MapReduceCoordinator`** — runs `CreditBureauAgent`, `IncomeVerifier`, and `FraudScreener` as parallel mappers, then feeds into `RiskAggregator` as the reducer.
2. **`AssemblyCoordinator`** — sequences the full pipeline: `ApplicationIngestor` → parallel analysis → `RiskAggregator`.

The `GuardrailSandwich` wraps each external-facing agent to protect against API failures (circuit-breaker pattern).

In [ ]:
# ── Wrap external agents in GuardrailSandwich (circuit-breaker pattern)
# The pre-filter validates we have required fields before calling the external service.
# The post-filter checks response integrity after the call.

def _pre_validate_step(ctx: Context) -> AgentStep:
    """Pre-guard: ensure application data is present before external call."""
    app = ctx.scratch.get('normalised_application', ctx.scratch.get('application', {}))
    valid = bool(app.get('application_id'))
    return AgentStep(
        out_messages=[Message(role='system', content=f'pre_validate: {"OK" if valid else "MISSING_APPLICATION"}')],
        state_updates={'pre_validate_ok': valid},
        stop=not valid,  # short-circuit if no application
    )

def _post_validate_step(ctx: Context) -> AgentStep:
    """Post-guard: confirm each sub-agent produced a risk score."""
    has_bureau = 'bureau_result' in ctx.scratch
    has_income = 'income_result' in ctx.scratch
    has_fraud  = 'fraud_result'  in ctx.scratch
    all_ok = has_bureau or has_income or has_fraud
    return AgentStep(
        out_messages=[Message(role='system', content=f'post_validate: bureau={has_bureau} income={has_income} fraud={has_fraud}')],
        state_updates={'post_validate_ok': all_ok},
        stop=False,
    )

pre_guard  = AgentBuilder('PreValidator',  'validator').with_step(_pre_validate_step).build()
post_guard = AgentBuilder('PostValidator', 'validator').with_step(_post_validate_step).build()

# Wrap the parallel analysis block
# Note: GuardrailSandwich wraps a single primary agent; we wrap the MapReduce block itself
guarded_bureau = GuardrailSandwich(
    prefilter=pre_guard,
    primary=bureau_agent,
    postfilter=post_guard,
)

# ── MapReduce: 3 mapper agents → 1 reducer (RiskAggregator)
parallel_analysis = MapReduceCoordinator(
    mappers=[bureau_agent, income_agent, fraud_agent],
    reducer=aggregator_agent,
)

# ── Wrap MapReduce inside AssemblyCoordinator for the full chain
# Stage 1: ingest, Stage 2: parallel analysis (via MapReduce)
# We'll run stages manually since AssemblyCoordinator uses Stage objects
print('Pipeline components assembled.')
print(f'  Parallel mappers: {[a.name for a in parallel_analysis.mappers]}')
print(f'  Reducer: {parallel_analysis.reducer.name}')

## Section 5 — Single Application Run

We run the full pipeline on Alice Chen (APP-001) and inspect every step's output.

In [ ]:
def run_credit_pipeline(applicant: Dict[str, Any], verbose: bool = True) -> Dict[str, Any]:
    """Run the full credit assessment pipeline on one applicant."""
    audit_trail = []
    
    # ── Step 1: Ingest
    ctx = Context(goal=f"Assess credit application {applicant['application_id']}")
    ctx.scratch['application'] = applicant
    
    ingest_result = ingestor_agent.run(ctx)
    audit_trail.append({'step': 'ingest', 'agent': 'ApplicationIngestor',
                        'output': json.loads(ingest_result.out_messages[-1].content)})
    
    # ── Step 2: Parallel analysis via MapReduceCoordinator
    # MapReduce runs each mapper against the shared context, then the reducer
    ctx.scratch['shards'] = [applicant['application_id']]  # single shard = full record
    analysis_result = parallel_analysis.run(goal=ctx.goal, inputs=ctx.scratch)
    
    # Collect intermediate results from context state updates
    for msg in analysis_result.messages:
        try:
            data = json.loads(msg.content)
            agent_name = data.get('agent', 'unknown')
            audit_trail.append({'step': 'analysis', 'agent': agent_name, 'output': data})
        except (json.JSONDecodeError, AttributeError):
            pass
    
    # Final aggregated result is the last message
    final_output = {}
    for msg in reversed(analysis_result.messages):
        try:
            data = json.loads(msg.content)
            if 'preliminary_decision' in data:
                final_output = data
                break
        except (json.JSONDecodeError, AttributeError):
            pass
    
    # ── Routing decision
    decision = final_output.get('preliminary_decision', 'MANUAL_REVIEW')
    audit_trail.append({'step': 'routing', 'decision': decision,
                        'confidence': final_output.get('decision_confidence', 0)})
    
    if verbose:
        print(f"\n{'='*60}")
        print(f"Application: {applicant['application_id']} — {applicant['name']}")
        print(f"Decision: {decision}")
        print(f"Confidence: {final_output.get('decision_confidence', 0):.2%}")
        print(f"Composite Risk: {final_output.get('composite_risk_score', 0):.4f}")
        print(f"Rationale: {final_output.get('rationale', 'N/A')}")
        print(f"Audit steps: {len(audit_trail)}")
    
    return {'final': final_output, 'audit_trail': audit_trail}


# Run on Alice Chen (expected: APPROVE)
result_alice = run_credit_pipeline(APPLICANTS[0])
print(f"\nFull audit trail ({len(result_alice['audit_trail'])} entries):")
for entry in result_alice['audit_trail']:
    print(f"  [{entry['step'].upper()}] {entry.get('agent', entry.get('decision', '?'))}")

## Section 6 — Batch Processing (5 Applications)

Process all five applicants and produce a summary table.

In [ ]:
batch_results = []

for applicant in APPLICANTS:
    result = run_credit_pipeline(applicant, verbose=False)
    final = result['final']
    batch_results.append({
        'id': applicant['application_id'],
        'name': applicant['name'],
        'credit_score': applicant['credit_score'],
        'income': applicant['annual_income'],
        'fraud_flags': applicant['fraud_flags'],
        'composite_risk': final.get('composite_risk_score', 'N/A'),
        'decision': final.get('preliminary_decision', 'N/A'),
        'confidence': final.get('decision_confidence', 0),
    })

print(f"{'ID':<8} {'Name':<14} {'Score':>6} {'Income':>8} {'Fraud':>5} {'Risk':>7} {'Decision':<14} {'Conf':>6}")
print('-' * 75)
for r in batch_results:
    print(f"{r['id']:<8} {r['name']:<14} {r['credit_score']:>6} ${r['income']:>7,} "
          f"{r['fraud_flags']:>5} {r['composite_risk']:>7.4f} {r['decision']:<14} {r['confidence']:>6.2%}")

## Section 7 — MCMC-Style Ensemble for Borderline Cases

For borderline applicants (composite risk between 0.35–0.45), we run the pipeline multiple times with slight perturbation (simulating MCMC sampling). This produces a distribution of scores from which we compute a stable consensus decision.

### Why MCMC for credit risk?

Credit risk models have **epistemic uncertainty** — the composite score is a point estimate, not a distribution. By sampling from that uncertainty region:

- We get a posterior distribution over the risk score
- The **mean** gives the expected risk
- The **standard deviation** quantifies decision uncertainty
- If >60% of chains agree on a decision → consensus is strong
- If chains are split → flag for mandatory human review

In [ ]:
def run_ensemble(applicant: Dict[str, Any], n_chains: int = 5, noise_scale: float = 0.02) -> Dict[str, Any]:
    """
    Run the credit pipeline n_chains times with Gaussian noise injected into
    intermediate scores, simulating MCMC sampling over the uncertainty region.
    
    Args:
        applicant: Application dictionary.
        n_chains: Number of MCMC chains (default=5).
        noise_scale: Std-dev of Gaussian noise added to risk scores.
        
    Returns:
        Dict with per-chain results, statistics, and consensus decision.
    """
    chain_results = []
    
    for chain_idx in range(n_chains):
        # Perturb the applicant data slightly to simulate sampling
        perturbed = copy.deepcopy(applicant)
        perturbed['credit_score'] = int(
            perturbed['credit_score'] + random.gauss(0, noise_scale * 50)
        )
        perturbed['annual_income'] = int(
            perturbed['annual_income'] * (1 + random.gauss(0, noise_scale))
        )
        
        result = run_credit_pipeline(perturbed, verbose=False)
        final = result['final']
        
        chain_results.append({
            'chain': chain_idx,
            'risk_score': final.get('composite_risk_score', 0.5),
            'decision': final.get('preliminary_decision', 'MANUAL_REVIEW'),
            'confidence': final.get('decision_confidence', 0.5),
        })
    
    # Compute statistics
    risk_scores = [c['risk_score'] for c in chain_results]
    decisions   = [c['decision']   for c in chain_results]
    confidences = [c['confidence'] for c in chain_results]
    
    mean_risk  = statistics.mean(risk_scores)
    std_risk   = statistics.stdev(risk_scores) if n_chains > 1 else 0.0
    
    decision_counts = {}
    for d in decisions:
        decision_counts[d] = decision_counts.get(d, 0) + 1
    
    consensus_decision = max(decision_counts, key=decision_counts.get)
    agreement_pct      = decision_counts[consensus_decision] / n_chains
    mean_confidence    = statistics.mean(confidences)
    
    return {
        'application_id': applicant['application_id'],
        'applicant_name': applicant['name'],
        'chains': chain_results,
        'mean_risk_score': round(mean_risk, 4),
        'std_risk_score': round(std_risk, 4),
        'consensus_decision': consensus_decision,
        'agreement_pct': round(agreement_pct, 4),
        'mean_confidence': round(mean_confidence, 4),
        'decision_counts': decision_counts,
        'requires_human_review': agreement_pct < 0.60 or mean_risk > 0.35,
    }


# Run ensemble on Eve Johnson (the borderline case)
ensemble_result = run_ensemble(APPLICANTS[4], n_chains=5)

print(f"\nEnsemble Analysis — {ensemble_result['applicant_name']}")
print(f"Chains run: {len(ensemble_result['chains'])}")
print(f"\nPer-chain results:")
for c in ensemble_result['chains']:
    print(f"  Chain {c['chain']}: risk={c['risk_score']:.4f}  decision={c['decision']:12s}  conf={c['confidence']:.2%}")

print(f"\nEnsemble Statistics:")
print(f"  Mean risk score : {ensemble_result['mean_risk_score']:.4f}")
print(f"  Std risk score  : {ensemble_result['std_risk_score']:.4f}")
print(f"  Consensus       : {ensemble_result['consensus_decision']}")
print(f"  Agreement       : {ensemble_result['agreement_pct']:.0%}")
print(f"  Mean confidence : {ensemble_result['mean_confidence']:.2%}")
print(f"  Human review    : {'YES — borderline case' if ensemble_result['requires_human_review'] else 'No'}")
print(f"  Decision counts : {ensemble_result['decision_counts']}")

## Section 8 — MCMC State Machine for Iterative Review

Borderline cases enter a three-state review cycle: **assess → review → decide**. The machine iterates until either:
- Confidence crosses the threshold (0.75), or  
- Maximum iterations (3) are reached

This mirrors how a loan officer would revisit a borderline file, each pass adding more data.

In [ ]:
from enum import Enum

class ReviewState(Enum):
    ASSESS  = 'assess'
    REVIEW  = 'review'
    DECIDE  = 'decide'
    DONE    = 'done'


def run_review_state_machine(
    applicant: Dict[str, Any],
    confidence_threshold: float = 0.75,
    max_iterations: int = 3,
) -> Dict[str, Any]:
    """
    3-state MCMC review machine.
    
    State transitions:
      ASSESS  → REVIEW  (if confidence < threshold)
      REVIEW  → DECIDE  (if consensus reached OR max iterations hit)
      DECIDE  → DONE
    """
    state     = ReviewState.ASSESS
    iteration = 0
    history   = []
    current_result = None
    n_chains   = 3

    while state != ReviewState.DONE and iteration < max_iterations:
        iteration += 1
        
        if state == ReviewState.ASSESS:
            # Initial assessment: single pipeline run
            current_result = run_credit_pipeline(applicant, verbose=False)['final']
            conf = current_result.get('decision_confidence', 0.5)
            history.append({'state': state.value, 'iteration': iteration,
                            'confidence': conf, 'action': 'initial_assessment'})
            state = ReviewState.REVIEW if conf < confidence_threshold else ReviewState.DECIDE
        
        elif state == ReviewState.REVIEW:
            # Ensemble review to improve confidence estimate
            ensemble = run_ensemble(applicant, n_chains=n_chains, noise_scale=0.015)
            conf = ensemble['mean_confidence']
            history.append({'state': state.value, 'iteration': iteration,
                            'confidence': conf, 'agreement': ensemble['agreement_pct'],
                            'action': f'ensemble_{n_chains}_chains'})
            current_result['preliminary_decision'] = ensemble['consensus_decision']
            current_result['decision_confidence']  = conf
            state = ReviewState.DECIDE if conf >= confidence_threshold else ReviewState.REVIEW
            n_chains = min(n_chains + 2, 9)  # expand chain count each iteration
        
        elif state == ReviewState.DECIDE:
            history.append({'state': state.value, 'iteration': iteration,
                            'decision': current_result.get('preliminary_decision'),
                            'action': 'final_decision'})
            state = ReviewState.DONE
    
    return {
        'final_decision': current_result.get('preliminary_decision', 'MANUAL_REVIEW'),
        'final_confidence': current_result.get('decision_confidence', 0),
        'iterations': iteration,
        'final_state': state.value,
        'history': history,
    }


# Run the state machine on Eve Johnson (borderline)
sm_result = run_review_state_machine(APPLICANTS[4], confidence_threshold=0.75)

print(f"State Machine Review — {APPLICANTS[4]['name']}")
print(f"\nState history:")
for step in sm_result['history']:
    conf_str = f"conf={step.get('confidence', 0):.2%}" if 'confidence' in step else ''
    agr_str  = f" agreement={step.get('agreement', 0):.0%}" if 'agreement' in step else ''
    dec_str  = f" decision={step.get('decision', '')}" if 'decision' in step else ''
    print(f"  [{step['state'].upper():8s}] iter={step['iteration']}  {conf_str}{agr_str}{dec_str}  action={step['action']}")

print(f"\nFinal Decision : {sm_result['final_decision']}")
print(f"Final Confidence: {sm_result['final_confidence']:.2%}")
print(f"Iterations used : {sm_result['iterations']}")

## Section 9 — Complete Audit Trail Output

Regulatory requirements (Basel III, SR 11-7, ECOA) mandate that every automated credit decision be fully explainable. Here we assemble the complete audit log for a declined application.

In [ ]:
def build_audit_report(applicant: Dict[str, Any]) -> Dict[str, Any]:
    """Build a complete regulatory audit report for one application."""
    pipeline_result = run_credit_pipeline(applicant, verbose=False)
    ensemble_result = run_ensemble(applicant, n_chains=5)
    sm_result       = run_review_state_machine(applicant, confidence_threshold=0.75)
    
    return {
        'report_metadata': {
            'generated_at': datetime.now(timezone.utc).isoformat(),
            'framework': 'Multigen / agentic_codex v0.1.0',
            'pipeline_version': '1.0',
            'regulatory_compliance': ['ECOA', 'FCRA', 'Basel III'],
        },
        'applicant': {
            'id': applicant['application_id'],
            'name': applicant['name'],
        },
        'pipeline_run': pipeline_result['final'],
        'ensemble_analysis': {
            'chains_run': len(ensemble_result['chains']),
            'mean_risk': ensemble_result['mean_risk_score'],
            'std_risk': ensemble_result['std_risk_score'],
            'consensus': ensemble_result['consensus_decision'],
            'agreement_pct': ensemble_result['agreement_pct'],
        },
        'review_machine': sm_result,
        'final_decision': sm_result['final_decision'],
        'agent_audit_trail': pipeline_result['audit_trail'],
    }


# Produce full audit report for Dave Patel (declined)
audit_report = build_audit_report(APPLICANTS[3])

print(json.dumps(audit_report, indent=2, default=str))

## Section 10 — Summary & Next Steps

### What we built

| Component | Pattern used | Purpose |
|-----------|-------------|----------|
| `ApplicationIngestor` | `AgentBuilder` + step fn | Data validation & normalisation |
| `CreditBureauAgent` | `AgentBuilder` + mock step | Credit history analysis |
| `IncomeVerifier` | `AgentBuilder` + mock step | Income stability check |
| `FraudScreener` | `AgentBuilder` + mock step | Fraud probability scoring |
| `RiskAggregator` | `AgentBuilder` + step fn | Weighted composite score |
| `MapReduceCoordinator` | Pipeline pattern | Parallel 3-way analysis |
| `GuardrailSandwich` | Safety pattern | Circuit-breaker protection |
| MCMC ensemble | Custom loop | Uncertainty quantification |
| State machine | Custom FSM | Iterative borderline review |
| Audit trail | Dict accumulation | Regulatory compliance |

### Production enhancements

1. **Replace `FunctionAdapter` steps** with `EnvOpenAIAdapter` + domain-specific prompts  
2. **Add persistent `EpisodicMemory`** to accumulate historical decisions per applicant  
3. **Connect `GuildRouter`** to real routing infrastructure (approve → origination system, decline → adverse action notice generator)  
4. **Instrument with `StructuredLogger`** and export to Prometheus for regulatory reporting  
5. **Add `TwoPersonRuleCoordinator`** for loans above $500k (dual-officer approval)

### Related notebooks

- `01_quickstart.ipynb` — getting started with the framework  
- `04_circuit_breakers.ipynb` — deep dive on resilience patterns  
- `06_fan_out_consensus.ipynb` — parallel reasoning and voting  
- `10_autonomy_transparency.ipynb` — epistemic confidence scoring